# Proyecto Clasificación de Textos 
## Parte 1: Machine Learning Clásico

Este Jupyter Notebook contiene el flujo de predicción para la competencia de aprendizaje de máquinas para predecir la década a la que pertenece un texto basándonos en los laboratorios de clase.

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
import warnings
warnings.filterwarnings("ignore")

import re 


### 1. Carga de los Datos
Revisamos los datos brindados en la competencia, `train.csv` y `eval.csv`. Asegúrate de que los archivos se encuentren en el mismo directorio del Notebook.

In [2]:
print("Cargando dataset...")
try:
    df_train = pd.read_csv("./data/train.csv")
    df_eval = pd.read_csv("./data/eval.csv")
    print("Datos cargados. Entrenamos con:", len(df_train), "registros.")
except FileNotFoundError:
    print("Error: Descarga los archivos train.csv y eval.csv acá primero.")


Cargando dataset...
Datos cargados. Entrenamos con: 31403 registros.


### 2. Procesamiento y Limpieza Textual (NLTK)
Siguiendo el laboratorio, utilizaremos tokenización, minúsculas, y truncaremos la palabra con Stemmer.

In [3]:
nltk.download("punkt")
nltk.download("punkt_tab")
stemmer = PorterStemmer()

def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    tokens = word_tokenize(str(texto).lower())
    return " ".join([stemmer.stem(t) for t in tokens if t.isalnum()])

print("Iniciando limpieza textual de Train y Eval...")
df_train["texto_limpio"] = df_train["text"].apply(limpiar_texto)
df_eval["texto_limpio"] = df_eval["text"].apply(limpiar_texto)
print("¡Textos procesados!")
df_train.head(3)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pifbl\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\pifbl\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Iniciando limpieza textual de Train y Eval...
¡Textos procesados!


,text,decade,texto_limpio
0,\nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de...,164,honoraria 57 pontem de 3 ste áqu len ne parec ...
1,"gone. Sus amigos , sus clientes, todo \ncuanto...",182,gone su amigo su client todo cuanto le rodea l...
2,"Prefosen quemanera,e per qualesfolpechas deuan...",157,prefosen quemanera e per qualesfolpecha deuan ...


In [4]:
# Opcion 2
print("Iniciando limpieza textual de Train y Eval...")
# Reemplazamos los signos raros y de puntuación por espacios
# Para el caractér "-" que indica la contuniadad de una pabra el la siguiente línea hacemos una excepción y lo eliminamos
df_train["clean_text"] = df_train["text"].str.replace(r"-\s*\n\s*","",regex=True)
df_train["clean_text"] = df_train["clean_text"].str.replace(r"[^a-zA-ZáéíóúÁÉÍÓÚñÑ0-9\s]"," ",regex=True)
df_eval["clean_text"] = df_eval["text"].str.replace(r"-\s*\n\s*","",regex=True)
df_eval["clean_text"] = df_eval["clean_text"].str.replace(r"[^a-zA-ZáéíóúÁÉÍÓÚñÑ0-9\s]"," ",regex=True)
# Arreglamos los espacios para que solo quede uno
df_train["clean_text"] = df_train["clean_text"].str.replace(r"\s+", " ", regex=True).str.strip()
df_eval["clean_text"] = df_eval["clean_text"].str.replace(r"\s+", " ", regex=True).str.strip()

def tokenizacion(text):
    return " ".join(word_tokenize(text))

df_train["clean_text"] = df_train["clean_text"].apply(tokenizacion)
df_eval["clean_text"] = df_eval["clean_text"].apply(tokenizacion)
print("¡Textos procesados!")
df_train.head(3)

Iniciando limpieza textual de Train y Eval...
¡Textos procesados!


,text,decade,texto_limpio,clean_text
0,\nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de...,164,honoraria 57 pontem de 3 ste áqu len ne parec ...,Honorarias jubiladas 57 dit ad Pontem de poref...
1,"gone. Sus amigos , sus clientes, todo \ncuanto...",182,gone su amigo su client todo cuanto le rodea l...,gone Sus amigos sus clientes todo cuanto le ro...
2,"Prefosen quemanera,e per qualesfolpechas deuan...",157,prefosen quemanera e per qualesfolpecha deuan ...,Prefosen quemanera e per qualesfolpechas deuan...


In [5]:
df_train["texto_limpio"][0]

'honoraria 57 pontem de 3 ste áqu len ne parec que aísilt o ayuda calsiodoro llama ocloso y 19'

In [6]:
df_train["clean_text"][0]

'Honorarias jubiladas 57 dit ad Pontem de poreft Proreg 118 3 9 M 70 pag 4 1 3 Ste ph Gratian difcept 291 áqu len nes parece que aísilte O ayuda Calsiodoro lib 6 epuft s 2 Donde llama Ocloso CINGV y 19'

### 3. Representación Vectorial (Vectorización TF-IDF)
Usando los conceptos de la clase, se usará `TfidfVectorizer` agregando bigramas y excluyendo stop words.

In [7]:
# max_features limita a las palabras más relevantes.
# ngram_range de 1 a 2 ayuda mucho en tareas clásicas.
stopwords_es = ["yo","mi","conmigo","tú","ti","contigo","vos","él","ella","ello","usted","sí","consigo","nosotros","nosotras","ellos",
                "ellas","ustedes","me","te","nos","lo","la","se","los","las","les","mía","mío","míos","mías","tuyo","tuya","tuyos","tuyas",
                "suyo","suya","suyos","suyas","nuestro","nuestra","nuestros","nuestras","este","esta","esto","estos","estas","ese","eso",
                "esa","esos","esas","aquel","aquello","aquella","aquellos","aquellas","uno","una","unos","unas","otro","otra","otros","otras",
                "cualquiera","cualesquiera","quienquiera","quienesquiera","demás","de","que","en","por","con","del","para","como","pues",
                "pero","porque","muy","mas","era","parte","donde","no","ni","ya","es","al","le","su","ha","si","sus","fue","quien","el","entre",
                "un","bien","dos","tu","don","tiempo"]

vectorizador = TfidfVectorizer(max_features=25000, stop_words=stopwords_es, ngram_range=(1,2))

print("Construyendo matriz TF-IDF...")
X = vectorizador.fit_transform(df_train["texto_limpio"])
y = df_train["decade"]

# Split local temporal para ver qué tan bueno es
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)


Construyendo matriz TF-IDF...


In [12]:
# Opción 2
stopwords_es = ["yo","mi","conmigo","tú","ti","contigo","vos","él","ella","ello","usted","sí","consigo","nosotros","nosotras","ellos",
                "ellas","ustedes","me","te","nos","lo","la","se","los","las","les","mía","mío","míos","mías","tuyo","tuya","tuyos","tuyas",
                "suyo","suya","suyos","suyas","nuestro","nuestra","nuestros","nuestras","este","esta","esto","estos","estas","ese","eso",
                "esa","esos","esas","aquel","aquello","aquella","aquellos","aquellas","uno","una","unos","unas","otro","otra","otros","otras",
                "cualquiera","cualesquiera","quienquiera","quienesquiera","demás","de","que","en","por","con","del","para","como","pues",
                "pero","porque","muy","mas","era","parte","donde","no","ni","ya","es","al","le","su","ha","si","sus","fue","quien","el","entre",
                "un","bien","dos","tu","don","tiempo"]

vectorizador = TfidfVectorizer(max_features=25000, stop_words=stopwords_es, max_df=0.8, ngram_range=(1,2))

print("Construyendo matriz TF-IDF...")
X_2 = vectorizador.fit_transform(df_train["clean_text"])
y_2 = df_train["decade"]
print("Matriz terminada")

# Split local temporal para ver qué tan bueno es
X_2_train, X_2_test, y_2_train, y_2_test = train_test_split(X_2, y_2, test_size=0.15, random_state=42)

Construyendo matriz TF-IDF...
Matriz terminada


### 4. Construcción del Modelo Clásico
Probaremos `SGDClassifier`, que suele dominar entre los sistemas clásicos para NLP (implementa Regresión Lineal/Logística robusta y rápida).

In [8]:
modelo_val = SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=2000, random_state=42)
modelo_val.fit(X_train, y_train)

y_pred_val = modelo_val.predict(X_test)
print("Métricas Locales (Opcional):")
print(classification_report(y_test, y_pred_val))
print("Accuracy Local:", accuracy_score(y_test, y_pred_val))


Métricas Locales (Opcional):
              precision    recall  f1-score   support

         150       0.36      0.66      0.47       116
         151       0.24      0.46      0.32       127
         152       0.38      0.59      0.46       137
         153       0.21      0.28      0.24       116
         154       0.31      0.44      0.37       126
         155       0.20      0.11      0.14       133
         156       0.35      0.36      0.35       116
         157       0.14      0.15      0.15       117
         158       0.19      0.17      0.18       115
         159       0.12      0.22      0.16       108
         160       0.08      0.05      0.06       130
         161       0.09      0.03      0.05       115
         162       0.25      0.17      0.20       136
         163       0.12      0.10      0.11       113
         164       0.09      0.08      0.08       119
         165       0.12      0.06      0.08       127
         166       0.04      0.01      0.01       11

In [13]:
# Opción 2
modelo_val_2 = SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=2000, random_state=42)
modelo_val_2.fit(X_2_train, y_2_train)

y_pred_val_2 = modelo_val_2.predict(X_2_test)
print("Métricas Locales (Opcional):")
print(classification_report(y_2_test, y_pred_val_2))
print("Accuracy Local:", accuracy_score(y_2_test, y_pred_val_2))

Métricas Locales (Opcional):
              precision    recall  f1-score   support

         150       0.38      0.66      0.48       116
         151       0.25      0.42      0.31       127
         152       0.41      0.57      0.48       137
         153       0.23      0.37      0.28       116
         154       0.35      0.46      0.39       126
         155       0.24      0.11      0.15       133
         156       0.38      0.38      0.38       116
         157       0.14      0.15      0.14       117
         158       0.21      0.21      0.21       115
         159       0.11      0.22      0.15       108
         160       0.13      0.09      0.11       130
         161       0.12      0.04      0.06       115
         162       0.28      0.15      0.20       136
         163       0.10      0.10      0.10       113
         164       0.12      0.10      0.11       119
         165       0.11      0.06      0.08       127
         166       0.08      0.01      0.02       11

### 5. Predicción Final y Respuesta Kaggle
Ajustamos nuevamente al total de datos para no perder esa información y exportamos.

In [ ]:
modelo_final = SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=2000, random_state=42)
modelo_final.fit(X, y)

print("Realizando predicciones finales...")
X_eval = vectorizador.transform(df_eval["texto_limpio"])
predicciones = modelo_final.predict(X_eval)

df_eval["answer"] = predicciones
submission = df_eval[["id", "answer"]]
submission.to_csv("submission.csv", index=False)
print("¡Archivo guardado como submission.csv! Listo para Kaggle.")


### 6. Guardar el Modelo (Requerimiento de la clase para Bloque Neón)

In [ ]:
print("Guardando modelos usando Joblib...")
joblib.dump(modelo_final, "modelo_clasificador.joblib")
joblib.dump(vectorizador, "vectorizador.joblib")
print("Archivo de modelos guardados (entregables válidos).")
